# CHẠY MÔ HÌNH FT TRÊN PUBLIC → DỰNG BÀI NỘP

Chạy SAU khi `richharness_run.ipynb` xong. Lấy `N_DOC`/`K_CHUNK` từ bảng N×k của nó.

## Nguyên tắc
Chấm **một lần** ở mức giàu nhất rồi **suy ra mọi cấu hình nhỏ hơn từ cùng bộ điểm** —
ô 3 xuất nhiều file nộp cùng lúc, không chấm lại lần nào. Quota 10 bài/ngày nên để LB chọn.

## Mốc phải vượt
Bài đang đứng: **precision 0.702 · recall 0.6712** (`max(ce,ce_deep)`, 50 văn bản, 2 góc nhìn).
Bài mới thắng khi **recall > 0.6712**.

## ⚠️ Sau khi nộp
Bảng xếp hạng hiển thị **bài mới nhất**, không phải bài tốt nhất.
Bài thăm dò thua thì **nộp lại `A_K50_max_DANGNOP.zip` ngay trong ngày.**

## ⚠️ 07-09: lần chạy trước HỎNG — đọc trước khi chạy lại
`find()` bắt được một file **trùng tên nhưng khác ruột** ở phiên resume.
500 câu đầu chấm trên rổ đúng (M20/K50, `max(ce,ce_deep)`), **500 câu sau chấm trên rổ của
`tang1_M10_K20`** (chỉ có `ce`, không có `ce_deep`) → rổ yếu hơn hẳn, 64/500 câu thậm chí
không chứa đáp án đang nộp. Ô 1 nay **kiểm vân tay bộ điểm bằng md5 + số ứng viên + số bản ghi
có `ce_deep`**, ô 2 vứt điểm cũ nếu cấu hình lệch, ô 3 chặn nộp nếu rổ không khớp `order`.

**Chỉ cần chạy lại 500 câu sau** (~1,5h): Add Input thêm dataset chứa
`public_ft_scores_SEED500.json` — ô 2 tự nạp nó rồi chấm tiếp từ câu 501.

## Cần Add Input
`project-ir` (có `scores_public_fusion_M20_K50_matchEmbedded.json`, `public-official.json`,
`selected-contexts`, `deep_chunk.py`) · dataset chứa mô hình FT.

In [ ]:
import os, sys, json, time, gc, zipfile, hashlib, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

assert torch.cuda.is_available(), (
    "CHUA BAT GPU. Panel ben phai -> Settings -> Accelerator -> GPU T4 x2, roi Run All lai. "
    "(Kaggle nap image CPU khi Accelerator = None, torch khong co CUDA.)")
print(f"  GPU: {torch.cuda.get_device_name(0)}")

# ===== SUA 2 SO NAY THEO BANG NxK CUA richharness =====
N_DOC, K_CHUNK = 3, 50    # k=50 = DOC TRON van ban (34,8 doan/van ban duoc truy hoi).
                          # LUU Y: KHONG suy duoc k nho hon tu bo diem nay. pick_chunks co dong
                          #   `if len(parts) <= k: return list(parts)` -> tra theo THU TU VAN BAN,
                          #   con khi len(parts) > k thi tra theo THU TU DIEM. Hai tap khac nhau.
                          # Chi truc N suy duoc. truc N PHANG; truc k CHUA BAO HOA.
# ======================================================
MAXLEN = 1024
BASE_MODEL = "AITeamVN/Vietnamese_Reranker"
OUT  = "/kaggle/working/public_ft_scores.json"
META = "/kaggle/working/public_ft_meta.json"
SUBDIR = "/kaggle/working/subs"; os.makedirs(SUBDIR, exist_ok=True)

# ===== VAN TAY BO DIEM DUNG (bat buoc) =====
# 07-09: lan chay truoc HONG vi find() bat duoc mot file TRUNG TEN nhung KHAC RUOT
#        o phien resume -> 500 cau sau duoc cham tren ro cua tang1_M10_K20 (khong co ce_deep).
#        Tu nay: nhan dien bo diem bang RUOT, khong bang TEN.
SIG_MD5   = "90e0935918d7ef96de04f9acf4a9ec6b"
SIG_NCAND = 50        # 50 ung vien/cau, dung cho ca 1000 cau
SIG_NDEEP = 20000     # so ban ghi co ce_deep phai >= muc nay (that su la 20000)
SIG_BYTES = 4137008   # kich thuoc dung — dung de loc truoc khi bam md5 (nhanh)
# ===========================================

ROOT="/kaggle/input"
def find_all(n):
    return [os.path.join(r,n) for r,_,fs in os.walk(ROOT) if n in fs]
def md5(p):
    h=hashlib.md5()
    with open(p,"rb") as f:
        for b in iter(lambda: f.read(1<<20), b""): h.update(b)
    return h.hexdigest()
def find(n, want=None):
    """want = md5 mong doi. Co nhieu ban trung ten -> CHON DUNG BAN, khong bat go dataset."""
    c=find_all(n)
    if not c: raise FileNotFoundError(f"KHONG THAY {n} duoi {ROOT}")
    if len(c)==1: return c[0]
    u={}
    for p in c: u.setdefault(md5(p),[]).append(p)
    if len(u)==1:
        print(f"  (i) {n}: {len(c)} ban trung ruot, dung ban dau"); return c[0]
    print(f"  (!) {n} co {len(u)} BAN KHAC RUOT:")
    for k,ps in u.items(): print(f"      {k}  {ps[0]}")
    if want and want in u:
        print(f"      -> chon ban khop van tay {want}")
        return u[want][0]
    raise RuntimeError(f"{n}: {len(u)} ban khac ruot va KHONG ban nao khop van tay {want}. "
                       "Go bot dataset, hoac sua SIG_MD5 neu that su da doi bo diem.")
def find_by_md5(want, nbytes, exts=(".json",)):
    """Tim file theo RUOT, khong theo TEN. Loc truoc bang kich thuoc nen rat nhanh:
       md5 giong nhau thi kich thuoc bat buoc giong nhau."""
    hits=[]
    for r,_,fs in os.walk(ROOT):
        for f in fs:
            if not f.endswith(exts): continue
            p=os.path.join(r,f)
            try:
                if os.path.getsize(p)!=nbytes: continue
            except OSError: continue
            if md5(p)==want: hits.append(p)
    return hits

def find_model_dir():
    cands=[r for r,_,fs in os.walk(ROOT)
           if "config.json" in fs and any(f.endswith((".safetensors",".bin"))
              and f.startswith(("model","pytorch_model")) for f in fs)]
    if not cands: raise FileNotFoundError("KHONG THAY model FT — da Add Input chua?")
    for c in cands:
        if os.path.basename(c)=="ft_listwise": return c
    for c in cands:
        if "finetune" in c.lower(): return c
    if len(cands)==1: return cands[0]
    raise FileNotFoundError(f"KHONG XAC DINH duoc model: {cands}")

FT      = find_model_dir()
TESTF   = find("public-official.json")
# 08-09: dataset project-ir chua mot ban KHAC RUOT duoi DUNG cai ten
#        scores_public_fusion_M20_K50_matchEmbedded.json (md5 7682dc56, ce_deep=10 000 = M10).
#        Ba lan lien tiep bi cai ten nay lua -> BO TIM THEO TEN. Tim theo RUOT.
_hit = find_by_md5(SIG_MD5, SIG_BYTES)
if _hit:
    SCORESF = _hit[0]
    print(f"  (v) tim thay bo diem dung theo md5, ten file la: {os.path.basename(SCORESF)}")
    if len(_hit)>1: print(f"      ({len(_hit)} ban giong het nhau, dung ban dau)")
else:
    print(f"  (!) KHONG co file nao trong {ROOT} co md5={SIG_MD5}")
    print(f"      Dang do lai theo ten (se hong o assert md5 ben duoi neu sai).")
    print(f"      CACH SUA: upload Ketqua_E/lai_ft/scores_public_M20K50_90e09359.json")
    print(f"      (4 137 008 byte) len dataset bat ky, TEN GI CUNG DUOC, roi cap nhat input.")
    for _n in ("scores_public_M20K50_90e09359.json",
               "scores_public_fusion_M20_K50_matchEmbedded.json"):
        _c=find_all(_n)
        if _c:
            for _p in _c: print(f"      thay {_p}  md5={md5(_p)}  {os.path.getsize(_p)} byte")
            SCORESF=_c[0]; break
    else: raise FileNotFoundError("Khong thay bo diem nao ca.")
UTIL    = os.path.dirname(find("deep_chunk.py"))
sys.path.insert(0, UTIL)
import deep_chunk as DC
DC.MERGE_CHARS = 1800
CTX=None
for _r,_,_fs in os.walk(ROOT):
    if any(f.startswith("context_") and f.endswith(".json") for f in _fs): CTX=_r; break
assert CTX and len(DC.read_passage(CTX, os.listdir(CTX)[0][8:-5]))>50, "CTX sai"
for n,v in [("FT",FT),("TEST",TESTF),("SCORES",SCORESF),("UTIL",UTIL),("CTX",CTX)]: print(f"  {n:<7} {v}")

test=json.load(open(TESTF,encoding="utf-8")); S=json.load(open(SCORESF,encoding="utf-8"))

# ----- KIEM VAN TAY: chan dung ngay neu bo diem khong phai ban that -----
SC_MD5 = md5(SCORESF)
_ncand = {len(v) for v in S.values()}
_ndeep = sum(1 for q in S for d in S[q] if "ce_deep" in S[q][d])
print(f"\n  bo diem md5={SC_MD5}  ung vien/cau={sorted(_ncand)}  ban ghi co ce_deep={_ndeep:,}")
assert len(S)==1000,                 f"bo diem co {len(S)} cau, phai la 1000"
assert _ncand=={SIG_NCAND},          f"ung vien/cau = {sorted(_ncand)}, phai la {SIG_NCAND} — SAI FILE"
assert _ndeep>=SIG_NDEEP,            f"chi {_ndeep:,} ban ghi co ce_deep (<{SIG_NDEEP:,}) — DAY LA BO DIEM TANG 1, SAI FILE"
assert SC_MD5==SIG_MD5, (
    f"md5 KHONG KHOP.\n  co  : {SC_MD5}\n  can : {SIG_MD5}\n"
    f"  file: {SCORESF}\n"
    "CACH SUA: upload Ketqua_E/lai_ft/scores_public_M20K50_90e09359.json len dataset, "
    "cap nhat input, chay lai. (Ban trong project-ir la M10, sai ro.)")
print("  van tay bo diem: DAT")
# -----------------------------------------------------------------------

Q=list(test)
assert set(S)==set(Q)==set(test), "qid khong khop de thi"
mx=lambda v: max(v["ce"], v.get("ce_deep",-9e9))
order={q:[d for d,_ in sorted(S[q].items(), key=lambda kv:-mx(kv[1]))] for q in Q}
CUR={q:order[q][0] for q in Q}          # = dung bai dang nop 0.702
print(f"\n  {len(Q)} cau · {SIG_NCAND} ung vien/cau")
print(f"  dung lai bai dang nop tu file diem: {len(CUR)} cau (moc precision 0.702 / recall 0.6712)")

# QUY TAC 6: DEM TRUOC KHI CHAM
t0=time.time(); n=0
for q in Q[:40]:
    for d in order[q][:N_DOC]: n+=len(DC.pick_chunks(test[q]["question"] if isinstance(test[q],dict) else test[q], CTX, d, k=K_CHUNK))
est=n/40*len(Q)
print(f"\n  uoc {est:,.0f} cap · ~{est/9.35/3600:.1f}h @9,35 cap/s · ~{est/4.7/3600:.1f}h neu cham 2x")
assert est < 250_000, f"qua nhieu ({est:,.0f}) — ha N_DOC hoac K_CHUNK"
tok = AutoTokenizer.from_pretrained(FT) if os.path.isfile(os.path.join(FT,"tokenizer.json")) \
      else AutoTokenizer.from_pretrained(BASE_MODEL)
qtext = lambda q: test[q]["question"] if isinstance(test[q],dict) else test[q]


In [ ]:
# ===== Cham mo hinh FT. Luu tung doan, moi 50 cau, chay lai la noi tiep. =====
# NOI TIEP AN TOAN. Nhan diem cu tu 3 nguon, theo thu tu uu tien:
#   (a) /kaggle/working/public_ft_scores.json  — phien nay da chay do
#   (b) public_ft_scores_SEED500.json trong input — 500 cau sach da loc san
#   (c) public_ft_scores.json trong input — BAN CU CON O NHIEM, o duoi tu vut cau hong
# Bat ke nguon nao: moi cau deu phai qua 2 cua kiem moi duoc giu lai.
src="working"
if not os.path.isfile(OUT):
    for nm,tag in [("public_ft_scores_SEED500.json","seed"),("public_ft_scores.json","ban cu")]:
        p=find_all(nm)
        if p:
            shutil.copy(p[0], OUT); src=tag
            print(f"  nap {tag}: {len(json.load(open(OUT)))} cau tu {p[0]}")
            break

res = json.load(open(OUT,encoding="utf-8")) if os.path.isfile(OUT) else {}
if res:
    n0=len(res)
    _m = json.load(open(META,encoding="utf-8")) if os.path.isfile(META) else {}
    if _m and not (_m.get("scores_md5")==SC_MD5 and _m.get("N_DOC")==N_DOC and _m.get("K_CHUNK")==K_CHUNK):
        print(f"  !! meta lech ({_m}) -> vut het, cham lai tu dau"); res={}
    else:
        # CUA 1: ro da cham phai dung bang order[:N_DOC], dung thu tu.
        bad=[q for q in res if list(res[q])!=order[q][:N_DOC]]
        for q in bad: res.pop(q)
        # CUA 2: so doan moi van ban phai khop pick_chunks(k=K_CHUNK) — bat lech K_CHUNK
        #        (cua 1 khong nhin thay duoc). Kiem 8 cau mau la du.
        chk=list(res)[:8]; lech=[]
        for q in chk:
            for d,v in res[q].items():
                n=len(DC.pick_chunks(qtext(q),CTX,d,k=K_CHUNK))
                if len(v)!=n and v!=[-9e9]: lech.append((q,d,len(v),n))
        if lech:
            print(f"  !! so doan khong khop K_CHUNK={K_CHUNK} (vd {lech[:2]}) -> vut het"); res={}
        else:
            print(f"  kiem {len(chk)} cau mau: so doan khop K_CHUNK={K_CHUNK}")
        if bad and res: print(f"  vut {len(bad)} cau co ro sai, giu lai {len(res)}")
    print(f"  nhan {len(res)}/{n0} cau tu nguon '{src}'")
json.dump({"scores_md5":SC_MD5,"N_DOC":N_DOC,"K_CHUNK":K_CHUNK}, open(META,"w"))
print(f"da co {len(res)}/{len(Q)} cau · con phai cham {len(Q)-len(res)} cau")

m = AutoModelForSequenceClassification.from_pretrained(FT).cuda().eval()
t0=time.time(); npair=0; done0=len(res)
with torch.no_grad():
    for i,q in enumerate(Q,1):
        if q in res: continue
        qt=qtext(q); per={}
        for d in order[q][:N_DOC]:
            ck=DC.pick_chunks(qt,CTX,d,k=K_CHUNK)
            if not ck: per[d]=[-9e9]; continue
            e=tok([qt]*len(ck), ck, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
            with torch.autocast("cuda",dtype=torch.float16):
                s=m(**{k:v.cuda() for k,v in e.items()}).logits.view(-1)
            per[d]=[float(x) for x in s]
            npair+=len(ck)
        res[q]=per
        if len(res)%50==0:
            json.dump(res, open(OUT,"w",encoding="utf-8"))
            el=time.time()-t0; dn=len(res)-done0
            print(f"  {len(res)}/{len(Q)} · {npair:,} cap · {el/60:.0f} phut · con ~{el/max(dn,1)*(len(Q)-len(res))/60:.0f} phut", flush=True)
json.dump(res, open(OUT,"w",encoding="utf-8"))
print(f"XONG · {npair:,} cap · {(time.time()-t0)/60:.0f} phut -> {OUT}")
print("TAI VE TRUOC KHI DONG PHIEN")
del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ===== Dung NHIEU bai nop tu CUNG bo diem — 0 giay GPU them =====
res=json.load(open(OUT,encoding="utf-8")); assert len(res)==len(Q), f"moi cham {len(res)}/{len(Q)}"
# CHOT LAI: ro da cham phai dung bang order[:N_DOC] o MOI cau, khong ngoai le.
_bad=[q for q in Q if list(res[q])!=order[q][:N_DOC]]
assert not _bad, f"{len(_bad)} cau cham nham ro (vd {_bad[:3]}) — KHONG DUOC NOP. Cham lai nhung cau do."
print(f"kiem ro: {len(Q)}/{len(Q)} cau khop order[:{N_DOC}]")

def pick(N,k=None):
    """k=None -> max tren TOAN BO doan da cham (dung). k so -> chi dung khi k >= K_CHUNK."""
    o={}
    for q in Q:
        best,bs=None,-9e9
        for d in order[q][:N]:
            v=res[q].get(d)
            if not v: continue
            s=max(v) if k is None else max(v[:k])
            if s>bs: bs,best=s,d
        o[q]=best or order[q][0]
    return o

CONFIGS=[(3,None),(2,None),(1,None)]   # CHI bien thien N. k=None = dung TOAN BO doan da cham.
print(f"{'cau hinh':<14}{'khac bai dang nop':>20}   ghi chu")
print("-"*62)
made=[]
for N,k in CONFIGS:
    p=pick(N,k)
    dif=sum(p[q]!=CUR[q] for q in Q)
    if N==1: assert dif==0, f"N=1 phai trung bai dang nop nhung khac {dif} cau — ro sai"
    sub={q:{"answer":[p[q]]} for q in Q}
    assert len(sub)==1000 and set(sub)==set(Q)
    assert all(len(v["answer"])==1 and isinstance(v["answer"][0],str) for v in sub.values())
    z=f"{SUBDIR}/sub_FT_N{N}_k{K_CHUNK}.zip"
    with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("submission.json", json.dumps(sub,ensure_ascii=False))
    note = "du thong tin" if dif>=80 else ("= bai dang nop, khong can nop" if dif==0 else "khac qua it")
    print(f"  N={N} k={K_CHUNK:<7}{dif:>15}/1000   {note}")
    made.append((z,dif))
print(f"\nDa dung {len(made)} bai -> {SUBDIR}/")
print("NOP theo thu tu: giau nhat truoc. Moc phai vuot: recall 0.6712 (= precision 0.702).")
print("Bai nao thua -> nop lai A_K50_max_DANGNOP.zip NGAY trong ngay (bang hien thi bai MOI NHAT).")
